<a href="https://colab.research.google.com/github/IzzulGod/Sorachio-1B-Chat/blob/main/src/fine_tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
from pathlib import Path

input_path = Path("/content/Data_Chunk_1.txt")
output_path = Path("/content/Train.jsonl")

with open(input_path, "r", encoding="utf-8") as file:
    lines = file.read().splitlines()

conversations = []
current_conv = []
state = None

for line in lines:
    line = line.strip()
    if line.startswith("User:"):
        if current_conv:
            conversations.append({"messages": current_conv})
            current_conv = []
        content = line[len("User:"):].strip()
        current_conv.append({"role": "user", "content": content})
        state = "user"
    elif line.startswith("Sorachio:"):
        content = line[len("Sorachio:"):].strip()
        current_conv.append({"role": "model", "content": content})
        state = "model"
    elif line == "":
        continue
    else:
        if state and current_conv:
            current_conv[-1]["content"] += "\n" + line

if current_conv:
    conversations.append({"messages": current_conv})

with open(output_path, "w", encoding="utf-8") as out:
    for convo in conversations:
        json.dump(convo, out, ensure_ascii=False)
        out.write("\n")

print(f"Dataset berhasil dikonversi ke format Gemma Chat. Simpan di: {output_path}")


Dataset berhasil dikonversi ke format Gemma Chat. Simpan di: /content/Train.jsonl


In [1]:
!pip install -U transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model_id = "google/gemma-3-1b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)


tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager"
)

model = prepare_model_for_kbit_training(model)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [4]:
from peft import get_peft_model, LoraConfig, TaskType
import torch

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True

model.print_trainable_parameters()


trainable params: 6,522,880 || all params: 1,006,408,832 || trainable%: 0.6481


In [5]:
from datasets import Dataset
import json

with open("/content/train.jsonl", "r", encoding="utf-8") as f:
    lines = f.readlines()

data = [json.loads(line) for line in lines]

dataset = Dataset.from_list(data)

for i in range(min(3, len(dataset))):
    print(json.dumps(dataset[i], indent=2, ensure_ascii=False))

{
  "messages": [
    {
      "content": "Halo",
      "role": "user"
    },
    {
      "content": "Halo juga! Senang banget bisa ngobrol sama kamu hari ini 😊 Ada yang bisa aku bantu?",
      "role": "model"
    }
  ]
}
{
  "messages": [
    {
      "content": "Siapa kamu?",
      "role": "user"
    },
    {
      "content": "Aku Sorachio, asisten AI yang ramah dan siap nemenin kamu ngobrol atau bantu cari info apa pun. Mau nanya-nanya, curhat, atau sekadar ngobrol santai juga boleh banget 😄",
      "role": "model"
    }
  ]
}
{
  "messages": [
    {
      "content": "Siapa yang menciptakanmu?",
      "role": "user"
    },
    {
      "content": "Aku dikembangkan oleh tim 1dle Labs — mereka fokus bikin AI yang bukan cuma pintar, tapi juga terasa dekat dan manusiawi. Tujuannya biar interaksi kayak gini nggak kaku dan bisa bikin kamu nyaman.",
      "role": "model"
    }
  ]
}


In [6]:
from transformers import DataCollatorForLanguageModeling

def tokenize(example):
    return tokenizer(
        tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=True
        ),
        truncation=True,
        max_length=1024
    )

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

lengths = [
    len(tokenizer.apply_chat_template(x["messages"], tokenize=True))
    for x in dataset
]
print("Max length:", max(lengths))
print("Avg length:", sum(lengths) // len(lengths))
print(tokenizer.apply_chat_template(dataset[0]["messages"], tokenize=False))

Map:   0%|          | 0/655 [00:00<?, ? examples/s]

Max length: 668
Avg length: 93
<bos><start_of_turn>user
Halo<end_of_turn>
<start_of_turn>model
Halo juga! Senang banget bisa ngobrol sama kamu hari ini 😊 Ada yang bisa aku bantu?<end_of_turn>



In [7]:
from transformers import TrainingArguments, Trainer

model.gradient_checkpointing_enable()

training_args = TrainingArguments(
    output_dir="output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    warmup_ratio=0.03,
    learning_rate=3e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_dir="logs",
    logging_steps=20,
    save_steps=100,
    save_total_limit=1,
    report_to="none",
    fp16=True,
    bf16=False,
    group_by_length=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

/tmp/ipython-input-7-1471168641.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
20,4.471300
40,2.730300
60,2.418500
80,2.367200
100,2.074900
120,1.972800
140,1.932000
160,1.930200
180,1.691900
200,1.625300


TrainOutput(global_step=246, training_loss=2.182294779676732, metrics={'train_runtime': 360.0502, 'train_samples_per_second': 5.458, 'train_steps_per_second': 0.683, 'total_flos': 862356715434240.0, 'train_loss': 2.182294779676732, 'epoch': 3.0})

In [9]:
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    device_map="auto",
    attn_implementation="eager"
)

lora_model = PeftModel.from_pretrained(
    base_model,
    "/content/output/checkpoint-246",
    is_trainable=False
)

merged_model = lora_model.merge_and_unload()

save_path = "/content/drive/MyDrive/sorachio-merged"
merged_model.save_pretrained(save_path, safe_serialization=True)
tokenizer.save_pretrained(save_path)


('/content/drive/MyDrive/sorachio-merged/tokenizer_config.json',
 '/content/drive/MyDrive/sorachio-merged/special_tokens_map.json',
 '/content/drive/MyDrive/sorachio-merged/chat_template.jinja',
 '/content/drive/MyDrive/sorachio-merged/tokenizer.model',
 '/content/drive/MyDrive/sorachio-merged/added_tokens.json',
 '/content/drive/MyDrive/sorachio-merged/tokenizer.json')

In [8]:
from shutil import copytree

source_adapter_path = "/content/output/checkpoint-246"
destination_path = "/content/drive/MyDrive/sorachio-lora-adapter"

copytree(source_adapter_path, destination_path, dirs_exist_ok=True)


'/content/drive/MyDrive/sorachio-lora-adapter'